In [0]:
import pandas as pd

In [0]:
%sql
create catalog if not exists 'global_partner_business_analysis_project' 

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS global_partner_business_analysis_project.source

In [0]:
%sql
drop schema if exists global_partner_business_analysis_project.global_partner_business_analysis

#### Uploading source files from google drive and saving as tables


In [0]:
%pip install gdown

In [0]:
%restart_python 

In [0]:
import gdown
import shutil

In [0]:
import pandas as pd

file_id = "1GXRZNgfngU6Yal6hzs5NClDgJoN3vEKZ"

url = f"https://drive.google.com/uc?id={file_id}"

local_path = "/tmp/order_items.csv"

# Download the file from Google Drive
gdown.download(url, local_path, quiet=False)

# Read CSV using pandas (pandas can access /tmp/ on serverless)
pdf = pd.read_csv(local_path)

# Convert to Spark DataFrame
df1 = spark.createDataFrame(pdf)

In [0]:
# Sanitize column names for Delta Lake
from pyspark.sql.functions import col
import re

def sanitize_column_name(name):
    # Replace invalid characters with underscores
    return re.sub(r'[ ,;{}()\n\t=]+', '_', name).strip('_')

df1_clean = df1.select([col(c).alias(sanitize_column_name(c)) for c in df1.columns])

df1_clean.write \
  .mode("overwrite") \
  .saveAsTable("`global_partner_business_analysis_project`.`source`.`order_items`")


In [0]:
file_id = "1l9anZqzpgTsQXe1ZTg-ihhn-9SBsa2H_"

url = f"https://drive.google.com/uc?id={file_id}"

local_path = "/tmp/order_item_options.csv"

# Download the file from Google Drive
gdown.download(url, local_path, quiet=False)

# Read CSV using pandas (pandas can access /tmp/ on serverless)
pdf = pd.read_csv(local_path)

# Convert to Spark DataFrame
df1 = spark.createDataFrame(pdf)

In [0]:
from pyspark.sql.functions import col
import re

def sanitize_column_name(name):
    # Replace invalid characters with underscores
    return re.sub(r'[ ,;{}()\n\t=]+', '_', name).strip('_')

df1_clean = df1.select([col(c).alias(sanitize_column_name(c)) for c in df1.columns])

df1_clean.write \
  .mode("overwrite") \
  .saveAsTable("`global_partner_business_analysis_project`.`source`.`order_item_options`")

In [0]:
file_id = "1v1rPl4nJp1B_nQmNm_Nrz2ZeBkppKRYh"

url = f"https://drive.google.com/uc?id={file_id}"

local_path = "/tmp/date_dim.csv"

# Download the file from Google Drive
gdown.download(url, local_path, quiet=False)

# Read CSV using pandas (pandas can access /tmp/ on serverless)
pdf = pd.read_csv(local_path)

# Convert to Spark DataFrame
df1 = spark.createDataFrame(pdf)

In [0]:
from pyspark.sql.functions import col
import re

def sanitize_column_name(name):
    # Replace invalid characters with underscores
    return re.sub(r'[ ,;{}()\n\t=]+', '_', name).strip('_')

df1_clean = df1.select([col(c).alias(sanitize_column_name(c)) for c in df1.columns])

df1_clean.write \
  .mode("overwrite") \
  .saveAsTable("`global_partner_business_analysis_project`.`source`.`date_dim`")

#### Exploring Each Table
 - Orders_items
 - Order_item_options
 - Date_dim


#### Table: Orders_items

In [0]:
%sql
select * from global_partner_business_analysis_project.source.order_items

In [0]:
df1 = spark.read.table("`global_partner_business_analysis_project`.`source`.`order_items`")

In [0]:
duplicates = df1.groupBy(df1.columns).count().filter("count > 1")

In [0]:
duplicates.display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count null values for each column
null_counts = df1.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df1.columns])
print(null_counts.collect()[0].asDict())

In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql import Row

# Count nulls in each column
null_counts = df1.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df1.columns
]).collect()[0].asDict()

# Convert dictionary to a DataFrame
null_df1 = spark.createDataFrame(
    [Row(column_name=k, null_count=v) for k, v in null_counts.items()]
)

display(null_df1)

In [0]:
null_df1.write \
    .mode("overwrite") \
    .saveAsTable("`global_partner_business_analysis_project`.`source`.`order_items_null_summary`")

In [0]:
%sql
select * from global_partner_business_analysis_project.source.order_items_null_summary

#### Table: Order_item_options

In [0]:
%sql
select * from global_partner_business_analysis_project.source.order_item_options

In [0]:
%sql
SELECT 
    ORDER_ID,
    COUNT(*) AS duplicate_count
FROM global_partner_business_analysis_project.source.order_item_options
GROUP BY
    ORDER_ID
HAVING COUNT(*) > 1;

In [0]:
df1 = spark.read.table("`global_partner_business_analysis_project`.`source`.`order_item_options`")

In [0]:
duplicates = df1.groupBy(df1.columns).count().filter("count > 1")

In [0]:
duplicates.display()

In [0]:
from pyspark.sql.functions import col

# Group duplicates by order_id
duplicate_df1 = df1.groupBy("order_id").count().filter(col("count") > 1)

display(duplicate_df1)

In [0]:
duplicate_df1.write \
    .mode("overwrite") \
    .saveAsTable("`global_partner_business_analysis_project`.`source`.`order_item_options_duplicates`")

In [0]:
%sql
select * from global_partner_business_analysis_project.source.order_item_options_duplicates

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count null values for each column
null_counts = df1.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df1.columns])
print(null_counts.collect()[0].asDict())

In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql import Row

# Count nulls in each column
null_counts = df1.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df1.columns
]).collect()[0].asDict()

# Convert dictionary to a DataFrame
null_df1 = spark.createDataFrame(
    [Row(column_name=k, null_count=v) for k, v in null_counts.items()]
)

display(null_df1)

In [0]:
null_df1.write \
    .mode("overwrite") \
    .saveAsTable("`global_partner_business_analysis_project`.`source`.`order_item_options_null_summary`")

In [0]:
%sql
select * from global_partner_business_analysis_project.source.order_item_options_null_summary

#### Table: Date_dim

In [0]:
df1 = spark.read.table("`global_partner_business_analysis_project`.`source`.`date_dim`")
display(df1)

In [0]:
%sql
select * from global_partner_business_analysis_project.source.date_dim

In [0]:
duplicates = df1.groupBy(df1.columns).count().filter("count > 1")
duplicates.display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count null values for each column
null_counts = df1.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df1.columns])
print(null_counts.collect()[0].asDict())

In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql import Row

# Count nulls in each column
null_counts = df1.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df1.columns
]).collect()[0].asDict()

# Convert dictionary to a DataFrame
null_df1 = spark.createDataFrame(
    [Row(column_name=k, null_count=v) for k, v in null_counts.items()]
)

display(null_df1)

In [0]:
null_df1.write \
    .mode("overwrite") \
    .saveAsTable("`global_partner_business_analysis_project`.`source`.`date_dim_null_summary`")

In [0]:
%sql
SELECT * FROM `global_partner_business_analysis_project`.`source`.`date_dim_null_summary`
